<a href="https://www.kaggle.com/code/avikdas567/predicting-student-test-scores?scriptVersionId=295172781" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv")
sub   = pd.read_csv("/kaggle/input/playground-series-s6e1/sample_submission.csv")

TARGET = "exam_score"
ID_COL = "id"

# -----------------------------
# Feature detection
# -----------------------------
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
features = [c for c in train.columns if c not in [TARGET, ID_COL]]

X = train[features]
y = train[TARGET]
X_test = test[features]

# -----------------------------
# Cross-validation setup
# -----------------------------
N_SPLITS = 3
SEEDS = [42, 2024]
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

test_preds = np.zeros(len(test))
TOTAL_RUNS = len(SEEDS) * N_SPLITS
step = 0

# -----------------------------
# Training loop
# -----------------------------
for seed in SEEDS:
    print(f"\nTraining seed {seed}")
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
        step += 1
        progress_pct = 100 * step / TOTAL_RUNS
        print(f"\n▶ Fold {step}/{TOTAL_RUNS}  (~{progress_pct:.0f}% done)")
        
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostRegressor(
            iterations=1800,
            learning_rate=0.04,
            depth=7,
            l2_leaf_reg=5,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=seed,
            task_type="GPU",
            devices="0",
            early_stopping_rounds=120,
            verbose=200
        )

        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_cols
        )

        test_preds += model.predict(X_test) / TOTAL_RUNS

# -----------------------------
# Save submission
# -----------------------------
sub["exam_score"] = test_preds
sub.to_csv("submission.csv", index=False)

print("\nsubmission.csv saved!")


Training seed 42

▶ Fold 1/6  (~17% done)
0:	learn: 18.4296273	test: 18.3759584	best: 18.3759584 (0)	total: 145ms	remaining: 4m 21s
200:	learn: 8.8288847	test: 8.8241106	best: 8.8241106 (200)	total: 7.8s	remaining: 1m 2s
400:	learn: 8.8030557	test: 8.8096795	best: 8.8096795 (400)	total: 15.2s	remaining: 53s
600:	learn: 8.7827041	test: 8.8005855	best: 8.8005855 (600)	total: 22.7s	remaining: 45.2s
800:	learn: 8.7654403	test: 8.7938813	best: 8.7938813 (800)	total: 30.2s	remaining: 37.7s
1000:	learn: 8.7495205	test: 8.7891624	best: 8.7891624 (1000)	total: 37.7s	remaining: 30.1s
1200:	learn: 8.7347909	test: 8.7855691	best: 8.7855691 (1200)	total: 45.3s	remaining: 22.6s
1400:	learn: 8.7210293	test: 8.7831844	best: 8.7831562 (1398)	total: 52.9s	remaining: 15.1s
1600:	learn: 8.7084890	test: 8.7812132	best: 8.7812132 (1600)	total: 1m	remaining: 7.52s
1799:	learn: 8.6960853	test: 8.7794053	best: 8.7794015 (1798)	total: 1m 8s	remaining: 0us
bestTest = 8.779401487
bestIteration = 1798
Shrink mode